# F2 · Tree, logistic and KNN on four boundary shapes (and after a monotonic transformation)

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Compare a tree (with `max_depth` and `min_samples_leaf` chosen in cross-validation), a logistic regression and a KNN on four boundary shapes in 2D (axis-parallel steps, XOR, diagonal, circle) and repeat the comparison after applying a **monotonic transformation** (`exp`) to the data: the tree does not change, the other two do.

## Dataset

`shapes()` (given): 1500 uniform points in [−3, 3]² with four labelings (`steps`, `XOR`, `diagonal`, `circle`) and 5% of swapped labels. 60/40 split (`random_state=0`). Models: tree with `GridSearchCV` (depth and minimum leaf size), `LogisticRegression` and `KNeighborsClassifier(15)` with `StandardScaler`.

## Educational purpose

Learn to choose the model from the **geometry of the boundary**: trees excel when the boundary is made of thresholds on single features (steps, XOR) and struggle with diagonals; the logistic is the opposite. And see the invariance of trees to monotonic transformations, which makes them independent of the scale and of the distribution of the features.

## Task

1. For each shape train the three models (the tree with `GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": [2, 3, 4, 5, 6, 8, None], "min_samples_leaf": [1, 5, 20]}, cv=cv)` with `cv = StratifiedKFold(5, shuffle=True, random_state=0)`) and save the test accuracy in `table` (index: shapes; columns `tree`, `logistic`, `knn`).
2. Save `depths` (dictionary shape → `max_depth` chosen by the search for the tree).
3. Repeat after the `exp` transformation applied to **all** the features (training and test): `exp_table` (same rows and columns; the tree with the hyperparameters already chosen).
4. Save `change` (`pd.DataFrame` with `exp_table − table`) and `max_tree_change` (maximum absolute value of the `tree` column), `max_other_change` (maximum absolute value of the other two columns).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


def shapes():
    rng = np.random.default_rng(0)
    n = 1500
    X = rng.uniform(-3, 3, (n, 2))
    labels = {
        "steps": ((X[:, 0] > 0.5) & (X[:, 1] < 1.0)) | ((X[:, 0] < -1.5) & (X[:, 1] > -0.5)),
        "XOR": (X[:, 0] > 0) ^ (X[:, 1] > 0),
        "diagonal": X[:, 0] + X[:, 1] > 0,
        "circle": (X ** 2).sum(1) < 4,
    }
    result = {}
    for name, y in labels.items():
        y = y.astype(int)
        y = np.where(rng.random(n) < 0.05, 1 - y, y)                    # 5% of wrong labels
        result[name] = (X, y)
    return result


data = shapes()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
grid = {"max_depth": [2, 3, 4, 5, 6, 8, None], "min_samples_leaf": [1, 5, 20]}

## Your work

In [ ]:
table = pd.DataFrame(index=list(data), columns=["tree", "logistic", "knn"], dtype=float)
depths = {}
best = {}
for name, (X, y) in data.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    ...
table.round(3)

In [ ]:
exp_table = pd.DataFrame(index=list(data), columns=["tree", "logistic", "knn"], dtype=float)
for name, (X, y) in data.items():
    X_tr, X_te, y_tr, y_te = train_test_split(np.exp(X), y, test_size=0.4, random_state=0)
    ...
exp_table.round(3)

In [ ]:
change = None
max_tree_change = None
max_other_change = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For each shape: `search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=cv).fit(X_tr, y_tr)`; `table.loc[name, "tree"] = search.score(X_te, y_te)`; `depths[name] = search.best_params_["max_depth"]`; `best[name] = search.best_params_`. With `exp`: `DecisionTreeClassifier(random_state=0, **best[name]).fit(...)`. Logistic and KNN: `make_pipeline(StandardScaler(), LogisticRegression())`, `make_pipeline(StandardScaler(), KNeighborsClassifier(15))`.'):
    check(table.notna().all().all() and set(depths) == set(data), "complete table and depth for each shape", "one row per shape")
    check((table.loc[["steps", "XOR", "circle"], "tree"] > table.loc[["steps", "XOR", "circle"], "logistic"] + 0.15).all() and table.loc["XOR", "logistic"] < 0.6, "the tree beats the logistic by over 15 points on steps, XOR and circle; on the XOR the logistic is at chance", "compare the columns")
    check(table.loc["diagonal", "logistic"] > table.loc["diagonal", "tree"] + 0.01 and table.loc["diagonal", "tree"] > 0.88, "on the diagonal the logistic beats the tree (which still goes over 0.88 with some depth)", "the diagonal is not parallel to the axes")
    check(depths["steps"] <= 5 and depths["diagonal"] is not None and depths["diagonal"] >= 4 and depths["circle"] >= depths["steps"], "chosen depth: the circle needs more depth than the steps; the diagonal needs at least 4", "best_params_['max_depth']")
    check(exp_table.notna().all().all(), "complete table with the exp transformation", "np.exp(X) for training and test")
    check(max_tree_change < 0.02 and max_other_change > 0.04, "after exp the tree changes by less than 2 points; logistic or KNN change by over 4 points", "change = exp_table - table")
    check(change.loc["diagonal", "logistic"] < -0.05, "the logistic on the diagonal gets worse by over 5 points with exp (the boundary is no longer a line)", "change['logistic']")
done()

---
The commented solution is in `solutions/F_compare/F2_boundary_shapes_and_invariance.ipynb`. Open it after you pass the check, or after you have really tried.